# 01. Data Preparation
## EmotionRadar | DM1590 KTH Final Project

This notebook:
1. Loads raw GoEmotions data (3 CSV files)
2. Filters out unclear examples
3. Maps 28 original emotion labels to 7 Ekman categories
4. Produces multi-label binary encoding (7 columns of 0/1)
5. Saves train/val/test splits to data/processed/

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [4]:
# Root of the repo
ROOT = Path("..") 

# Raw CSVs
RAW_DIR = ROOT / "data" / "raw"

# Processed output
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# Load files and merge into one
raw_data = pd.concat([
    pd.read_csv(RAW_DIR / "goemotions_1.csv"),
    pd.read_csv(RAW_DIR / "goemotions_2.csv"),
    pd.read_csv(RAW_DIR / "goemotions_3.csv")
], ignore_index=True)


In [7]:
# Remove example_very_unclear is not False
raw_data = raw_data[raw_data["example_very_unclear"] == False].reset_index(drop=True)


In [9]:
# Ekman taxonomy mapping (Demszky et al., 2020 — GoEmotions paper)
ekman_mapping = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude",
                 "love", "optimism", "relief", "pride", "admiration", 
                 "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"]
}

print("Ekman categories:", list(ekman_mapping.keys()))

Ekman categories: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


In [12]:
emotion_cols = list(ekman_mapping.keys()) + [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

df = pd.DataFrame()
df["text"] = raw_data["text"]

for ekman_label, source_labels in ekman_mapping.items():
    # Keep only source labels that actually exist as columns
    valid_sources = [col for col in source_labels if col in raw_data.columns]
    # If ANY source label is 1 → this Ekman label is 1
    df[ekman_label] = raw_data[valid_sources].any(axis=1).astype(int)

print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (211225, 8)

Column names: ['text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']

First 5 rows:


,text,anger,disgust,fear,joy,sadness,surprise,neutral
0,That game hurt.,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",0,0,0,0,0,0,1
3,Man I love reddit.,0,0,0,1,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",0,0,0,0,0,0,1


In [13]:
# Every row should have at least 1 emotion (sum across emotion columns >= 1)
emotion_columns = list(ekman_mapping.keys())
row_sums = df[emotion_columns].sum(axis=1)

print("=== Sanity Checks ===")
print(f"Total rows: {len(df)}")
print(f"Rows with zero emotions (should be 0): {(row_sums == 0).sum()}")
print(f"Rows with multiple emotions: {(row_sums > 1).sum()}")
print(f"\nEmotion distribution:")
print(df[emotion_columns].sum().sort_values(ascending=False))

=== Sanity Checks ===
Total rows: 211225
Rows with zero emotions (should be 0): 3411
Rows with multiple emotions: 18024

Emotion distribution:
joy         82938
neutral     55298
anger       30473
surprise    29282
sadness     19101
disgust      5301
fear         4515
dtype: int64


In [14]:
df.to_csv(PROCESSED_DIR / "goemotions_ekman.csv", index=False)
print("Saved to:", PROCESSED_DIR / "goemotions_ekman.csv")
print("Shape:", df.shape)

Saved to: ../data/processed/goemotions_ekman.csv
Shape: (211225, 8)
